# 04 — Prediction of Individual Brain Network Reorganization

This notebook evaluates whether pre-neurofeedback resting-state
functional connectivity can predict individual post-neurofeedback
network reorganization.

The prediction target is the participant-specific connectivity change:

\[
\Delta G = G_{\mathrm{Rest2}} - G_{\mathrm{Rest1}}
\]

Because the current sample is small, prediction begins with
low-dimensional network-level features and regularized models rather
than high-capacity graph neural networks.

## 1 — Prediction Target

The first prediction task evaluates whether baseline Rest1 functional
connectivity can predict the magnitude of subsequent whole-brain
network reorganization.

Predictors:

\[
X = \text{Rest1 network-level connectivity features}
\]

Target:

\[
y = \text{Mean Absolute } \Delta FC
\]

where larger target values indicate greater overall pre-to-post
functional connectivity reorganization.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from itertools import combinations_with_replacement
from nilearn import datasets

# -----------------------------
# Input directories
# -----------------------------

ZFC_DIR = Path("outputs") / "functional_connectivity_216_fisherz"

# Same 18 QC-passing participants
SUBJECTS = [
    "E3746", "E3799", "E3973", "E4051",
    "E4209", "E4253", "E4324", "E4350",
    "E4360", "E4484", "E4689", "E4697",
    "E4745", "E5215", "E5580", "E5586",
    "E5693", "E5694"
]

# -----------------------------
# Reconstruct network mapping
# -----------------------------

schaefer_info = datasets.fetch_atlas_schaefer_2018(
    n_rois=200,
    yeo_networks=7,
    resolution_mm=2
)

schaefer_labels = schaefer_info.labels[1:]

schaefer_labels = [
    label.decode("utf-8") if isinstance(label, bytes) else str(label)
    for label in schaefer_labels
]

schaefer_networks = [
    label.split("_")[2]
    for label in schaefer_labels
]

roi_networks = (
    schaefer_networks +
    ["Subcortical"] * 16
)

roi_networks_array = np.array(roi_networks)

network_order = [
    "Vis",
    "SomMot",
    "DorsAttn",
    "SalVentAttn",
    "Limbic",
    "Cont",
    "Default",
    "Subcortical"
]

network_pairs = list(
    combinations_with_replacement(
        network_order, 2
    )
)

print("ROIs:", len(roi_networks))
print("Network pairs/features:", len(network_pairs))
print("Participants:", len(SUBJECTS))

In [ ]:
rest1_files = [
    ZFC_DIR / f"{subject}_Rest1_FC_Z_216.csv"
    for subject in SUBJECTS
]

missing_files = [
    str(path)
    for path in rest1_files
    if not path.exists()
]

print("Rest1 files found:", len(rest1_files) - len(missing_files))
print("Expected:", 18)

print("\nMissing files:")
print(missing_files)

In [ ]:
feature_rows = []

for subject in SUBJECTS:

    fc_file = ZFC_DIR / f"{subject}_Rest1_FC_Z_216.csv"

    fc = pd.read_csv(
        fc_file,
        index_col=0
    ).to_numpy()

    row = {
        "Subject": subject
    }

    for net1, net2 in network_pairs:

        idx1 = np.where(
            roi_networks_array == net1
        )[0]

        idx2 = np.where(
            roi_networks_array == net2
        )[0]

        block = fc[
            np.ix_(idx1, idx2)
        ].copy()

        # Remove self-connections for within-network features
        if net1 == net2:
            np.fill_diagonal(block, np.nan)

        feature_name = f"{net1}__{net2}"

        row[feature_name] = np.nanmean(block)

    feature_rows.append(row)


X_df = pd.DataFrame(feature_rows)

print("Feature matrix shape:", X_df.shape)
print("Participants:", X_df["Subject"].nunique())
print("Connectivity features:", X_df.shape[1] - 1)

print("\nMissing feature values:")
print(X_df.isna().sum().sum())

display(X_df.head())

## 2 — Reorganization Target

The prediction target is each participant's magnitude of whole-brain
functional connectivity reorganization.

For each participant, the absolute Fisher-z connectivity change is
averaged across all unique off-diagonal connections:

\[
y_i = \mathrm{mean}\left(|Z_{Rest2,i} - Z_{Rest1,i}|\right)
\]

Higher values indicate greater overall functional network
reorganization between Rest1 and Rest2.

In [ ]:
ZDELTA_DIR = Path("outputs") / "delta_connectivity_216_fisherz"

target_rows = []

upper_triangle = np.triu_indices(
    216,
    k=1
)

for subject in SUBJECTS:

    delta_file = (
        ZDELTA_DIR /
        f"{subject}_DeltaFC_Z_216.csv"
    )

    delta = pd.read_csv(
        delta_file,
        index_col=0
    ).to_numpy()

    # Unique off-diagonal connections only
    edge_values = delta[upper_triangle]

    mean_absolute_delta_z = np.nanmean(
        np.abs(edge_values)
    )

    target_rows.append({
        "Subject": subject,
        "Mean_Absolute_DeltaZ": mean_absolute_delta_z
    })


y_df = pd.DataFrame(target_rows)

print("Target participants:", len(y_df))

print("\nMissing target values:")
print(y_df["Mean_Absolute_DeltaZ"].isna().sum())

print("\nTarget summary:")
print(
    y_df["Mean_Absolute_DeltaZ"].describe()
)

display(y_df)

## 3 — Prediction Dataset

Baseline network-level connectivity features are merged with the
participant-specific functional connectivity reorganization target.

Each row represents one participant, with 36 Rest1 network-connectivity
features and one subsequent reorganization target.

In [ ]:
prediction_df = X_df.merge(
    y_df,
    on="Subject",
    how="inner"
)

feature_columns = [
    col for col in X_df.columns
    if col != "Subject"
]

X = prediction_df[
    feature_columns
].to_numpy()

y = prediction_df[
    "Mean_Absolute_DeltaZ"
].to_numpy()

print("Participants:", len(prediction_df))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nMissing X values:", np.isnan(X).sum())
print("Missing y values:", np.isnan(y).sum())

print("\nSubject order:")
print(prediction_df["Subject"].tolist())

## 4 — Ridge Regression Baseline

A regularized linear regression model is used as the initial
prediction baseline because the number of baseline connectivity
features exceeds the number of participants.

Leave-one-out cross-validation (LOOCV) is used so that each
participant is predicted from a model trained only on the remaining
participants.

Feature standardization is performed within each training fold to
avoid information leakage.

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

loo = LeaveOneOut()

y_true = []
y_pred = []

for train_idx, test_idx in loo.split(X):

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    model.fit(
        X_train,
        y_train
    )

    prediction = model.predict(
        X_test
    )[0]

    y_true.append(
        y_test[0]
    )

    y_pred.append(
        prediction
    )


y_true = np.array(y_true)
y_pred = np.array(y_pred)

mae = mean_absolute_error(
    y_true,
    y_pred
)

r, p = pearsonr(
    y_true,
    y_pred
)

# Baseline: always predict training-sample mean
baseline_predictions = []

for train_idx, test_idx in loo.split(X):
    baseline_predictions.append(
        np.mean(y[train_idx])
    )

baseline_mae = mean_absolute_error(
    y,
    baseline_predictions
)

print("RIDGE LOOCV")
print("MAE:", round(mae, 4))
print("Pearson r:", round(r, 4))
print("Correlation p:", round(p, 4))

print("\nMEAN BASELINE")
print("MAE:", round(baseline_mae, 4))

print("\nImprovement over baseline:",
      round(baseline_mae - mae, 4))

## 5 — Nested Cross-Validated Ridge Regression

To avoid selecting the regularization parameter using held-out
participants, Ridge regularization strength is selected independently
within each outer leave-one-out training fold.

The outer loop estimates out-of-sample prediction performance, while
the inner cross-validation loop selects the Ridge alpha using only
the training participants.

In [ ]:
from sklearn.model_selection import GridSearchCV

outer_loo = LeaveOneOut()

alpha_grid = {
    "ridge__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100,
        1000
    ]
}

nested_true = []
nested_pred = []
selected_alphas = []

for train_idx, test_idx in outer_loo.split(X):

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge())
    ])

    # Inner LOOCV uses only the 17 training participants
    inner_loo = LeaveOneOut()

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=alpha_grid,
        cv=inner_loo,
        scoring="neg_mean_absolute_error"
    )

    search.fit(
        X_train,
        y_train
    )

    prediction = search.predict(
        X_test
    )[0]

    nested_true.append(
        y_test[0]
    )

    nested_pred.append(
        prediction
    )

    selected_alphas.append(
        search.best_params_["ridge__alpha"]
    )


nested_true = np.array(nested_true)
nested_pred = np.array(nested_pred)

nested_mae = mean_absolute_error(
    nested_true,
    nested_pred
)

nested_r, nested_p = pearsonr(
    nested_true,
    nested_pred
)

print("NESTED RIDGE LOOCV")
print("MAE:", round(nested_mae, 4))
print("Pearson r:", round(nested_r, 4))
print("Correlation p:", round(nested_p, 4))

print("\nMEAN BASELINE")
print("MAE:", round(baseline_mae, 4))

print(
    "\nImprovement over baseline:",
    round(baseline_mae - nested_mae, 4)
)

print("\nSelected alphas:")
print(selected_alphas)

print("\nAlpha frequencies:")
print(
    pd.Series(selected_alphas)
      .value_counts()
      .sort_index()
)

In [ ]:
prediction_results = pd.DataFrame({
    "Subject": prediction_df["Subject"],
    "Observed_DeltaZ": nested_true,
    "Predicted_DeltaZ": nested_pred,
    "Residual": nested_true - nested_pred,
    "Selected_Alpha": selected_alphas
})

display(prediction_results)

print("\nObserved mean:",
      round(prediction_results["Observed_DeltaZ"].mean(), 4))

print("Predicted mean:",
      round(prediction_results["Predicted_DeltaZ"].mean(), 4))

print("Largest absolute prediction error:")
display(
    prediction_results
    .assign(
        Absolute_Error=lambda x: np.abs(x["Residual"])
    )
    .sort_values(
        "Absolute_Error",
        ascending=False
    )
    .head(5)
)

## 6 — Network-Specific Reorganization Targets

Whole-brain mean absolute change compresses each participant's
reorganization into a single scalar and may discard information about
the spatial pattern of network change.

Therefore, a second analysis represents each participant's
Rest1-to-Rest2 transition using 36 within- and between-network
connectivity changes.

For each network pair:

\[
\Delta Z_{ab} = Z_{Rest2,ab} - Z_{Rest1,ab}
\]

This produces a 36-dimensional network-reorganization profile for
each participant.

In [ ]:
delta_feature_rows = []

for subject in SUBJECTS:

    delta_file = (
        ZDELTA_DIR /
        f"{subject}_DeltaFC_Z_216.csv"
    )

    delta = pd.read_csv(
        delta_file,
        index_col=0
    ).to_numpy()

    row = {
        "Subject": subject
    }

    for net1, net2 in network_pairs:

        idx1 = np.where(
            roi_networks_array == net1
        )[0]

        idx2 = np.where(
            roi_networks_array == net2
        )[0]

        block = delta[
            np.ix_(idx1, idx2)
        ].copy()

        if net1 == net2:
            np.fill_diagonal(
                block,
                np.nan
            )

        feature_name = (
            f"Delta_{net1}__{net2}"
        )

        row[feature_name] = np.nanmean(
            block
        )

    delta_feature_rows.append(row)


delta_network_df = pd.DataFrame(
    delta_feature_rows
)

print(
    "Delta network matrix shape:",
    delta_network_df.shape
)

print(
    "Participants:",
    delta_network_df["Subject"].nunique()
)

print(
    "Network-change features:",
    delta_network_df.shape[1] - 1
)

print(
    "Missing values:",
    delta_network_df.isna().sum().sum()
)

display(delta_network_df.head())

## 7 — Dimensionality of Network Reorganization

The 36 network-level connectivity changes are highly multivariate
relative to the sample size. Principal component analysis (PCA) is
first used descriptively to determine whether individual
reorganization profiles can be represented by a smaller number of
dominant transition patterns.

This descriptive analysis is not used to estimate predictive
performance. PCA used for prediction will subsequently be fitted
within cross-validation folds to avoid information leakage.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

Y_delta = (
    delta_network_df
    .drop(columns="Subject")
    .to_numpy()
)

pca = PCA()
pca.fit(Y_delta)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

print("Y_delta shape:", Y_delta.shape)

print("\nVariance explained:")
for i in range(min(10, len(explained))):
    print(
        f"PC{i+1}: "
        f"{explained[i]:.3f} "
        f"(cumulative {cumulative[i]:.3f})"
    )

print(
    "\nComponents needed for 80% variance:",
    np.argmax(cumulative >= 0.80) + 1
)

print(
    "Components needed for 90% variance:",
    np.argmax(cumulative >= 0.90) + 1
)

plt.figure(figsize=(7, 4))

plt.plot(
    range(1, len(cumulative) + 1),
    cumulative,
    marker="o"
)

plt.axhline(
    0.80,
    linestyle="--"
)

plt.axhline(
    0.90,
    linestyle="--"
)

plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title(
    "Dimensionality of Individual Network Reorganization"
)

plt.tight_layout()
plt.show()

## 8 — Dominant Reorganization Pattern

PC1 explains the largest proportion of variance in individual
network-reorganization profiles. Its loadings are examined to
determine which within- and between-network connectivity changes
contribute most strongly to this dominant transition pattern.

In [ ]:
delta_feature_names = (
    delta_network_df
    .drop(columns="Subject")
    .columns
)

pc1_loadings = pd.DataFrame({
    "Feature": delta_feature_names,
    "PC1_Loading": pca.components_[0]
})

pc1_loadings["Absolute_Loading"] = (
    pc1_loadings["PC1_Loading"].abs()
)

pc1_loadings = (
    pc1_loadings
    .sort_values(
        "Absolute_Loading",
        ascending=False
    )
)

print(
    "PC1 variance explained:",
    round(explained[0], 3)
)

print("\nTop 15 PC1 loadings:")

display(
    pc1_loadings.head(15)
)

In [ ]:
print("Positive PC1 loadings:",
      (pca.components_[0] > 0).sum())

print("Negative PC1 loadings:",
      (pca.components_[0] < 0).sum())

print("\nPC1 loading range:")
print(
    "Minimum:",
    round(pca.components_[0].min(), 4)
)
print(
    "Maximum:",
    round(pca.components_[0].max(), 4)
)

print("\nLowest 10 PC1 loadings:")

display(
    pd.DataFrame({
        "Feature": delta_feature_names,
        "PC1_Loading": pca.components_[0]
    })
    .sort_values("PC1_Loading")
    .head(10)
)

## 9 — Individual Dominant Transition Scores

Each participant is projected onto the dominant reorganization
component (PC1). Positive and negative scores represent opposite
directions along the distributed whole-brain connectivity transition
axis.

In [ ]:
pca_scores = pca.transform(Y_delta)

pc1_scores_df = pd.DataFrame({
    "Subject": delta_network_df["Subject"],
    "PC1_Score": pca_scores[:, 0]
})

pc1_scores_df = (
    pc1_scores_df
    .sort_values(
        "PC1_Score",
        ascending=False
    )
)

display(pc1_scores_df)

print("\nPC1 score summary:")
print(
    pc1_scores_df["PC1_Score"].describe()
)

print(
    "\nPositive scores:",
    (pc1_scores_df["PC1_Score"] > 0).sum()
)

print(
    "Negative scores:",
    (pc1_scores_df["PC1_Score"] < 0).sum()
)

## 10 — Dominant Transition Pattern by Neurofeedback Group

Before using the dominant transition score as a prediction target,
its relationship with intervention group is examined.

This determines whether the dominant Rest1-to-Rest2 connectivity
transition primarily reflects treatment-group differences or
individual variability across participants.

In [ ]:
PARTICIPANTS_FILE = Path(
    r"C:\Neurofeedback_Data_Fall26\Imaging\Processed Rest1&2 Scans\participants.tsv"
)

participants = pd.read_csv(
    PARTICIPANTS_FILE,
    sep="\t"
)

print("Participants file loaded:", PARTICIPANTS_FILE.exists())
print("Shape:", participants.shape)
print("Exam column exists:", "Exam" in participants.columns)
print("Group column exists:", "Group" in participants.columns)

In [ ]:
# Get one group label per participant
group_df = (
    participants[
        ["Exam", "Group"]
    ]
    .drop_duplicates("Exam")
    .rename(columns={"Exam": "Subject"})
)

pc1_group_df = (
    pc1_scores_df
    .merge(
        group_df,
        on="Subject",
        how="left"
    )
)

print("Missing group labels:",
      pc1_group_df["Group"].isna().sum())

print("\nGroup counts:")
print(
    pc1_group_df["Group"].value_counts()
)

print("\nPC1 by group:")
print(
    pc1_group_df
    .groupby("Group")["PC1_Score"]
    .agg(["count", "mean", "std"])
)

In [ ]:
from scipy.stats import ttest_ind

active_pc1 = (
    pc1_group_df.loc[
        pc1_group_df["Group"] == "active",
        "PC1_Score"
    ].to_numpy()
)

sham_pc1 = (
    pc1_group_df.loc[
        pc1_group_df["Group"] == "sham",
        "PC1_Score"
    ].to_numpy()
)

# Welch independent-samples t-test
t_stat, p_value = ttest_ind(
    active_pc1,
    sham_pc1,
    equal_var=False
)

# Cohen's d
n1 = len(active_pc1)
n2 = len(sham_pc1)

s1 = np.std(active_pc1, ddof=1)
s2 = np.std(sham_pc1, ddof=1)

pooled_sd = np.sqrt(
    ((n1 - 1) * s1**2 +
     (n2 - 1) * s2**2)
    /
    (n1 + n2 - 2)
)

cohens_d = (
    np.mean(active_pc1) -
    np.mean(sham_pc1)
) / pooled_sd

print("PC1 GROUP COMPARISON")

print("\nActive")
print("n:", n1)
print("Mean:", round(np.mean(active_pc1), 4))
print("SD:", round(s1, 4))

print("\nSham")
print("n:", n2)
print("Mean:", round(np.mean(sham_pc1), 4))
print("SD:", round(s2, 4))

print("\nActive - Sham difference:",
      round(
          np.mean(active_pc1) -
          np.mean(sham_pc1),
          4
      ))

print("Welch t:", round(t_stat, 4))
print("p-value:", round(p_value, 4))
print("Cohen's d:", round(cohens_d, 4))

## 11 — Intervention-Aware Predictors

Because post-neurofeedback brain-state transitions may depend on both
baseline network organization and the intervention received,
neurofeedback group is incorporated as an additional predictor.

The resulting formulation is:

\[
f(G_{\mathrm{Rest1}}, I) \rightarrow \Delta G
\]

where \(G_{\mathrm{Rest1}}\) represents baseline network organization
and \(I\) represents active versus sham neurofeedback.

In [ ]:
group_lookup = (
    pc1_group_df[
        ["Subject", "Group"]
    ]
    .drop_duplicates("Subject")
)

prediction_group_df = (
    prediction_df
    .merge(
        group_lookup,
        on="Subject",
        how="left"
    )
)

# Binary intervention indicator
prediction_group_df["Active"] = (
    prediction_group_df["Group"]
    .eq("active")
    .astype(int)
)

X_group = prediction_group_df[
    feature_columns + ["Active"]
].to_numpy()

print("Participants:", len(prediction_group_df))
print("Baseline FC features:", len(feature_columns))
print("Intervention features: 1")
print("X_group shape:", X_group.shape)

print("\nGroup coding:")
print(
    prediction_group_df[
        ["Subject", "Group", "Active"]
    ]
)

print(
    "\nMissing predictor values:",
    np.isnan(X_group).sum()
)

## 12 — Leakage-Free Prediction of the Dominant Brain-State Transition

Prediction of the dominant network-reorganization pattern is evaluated
using an outer leave-one-out procedure.

For each held-out participant, PCA is fitted exclusively to the
reorganization profiles of the remaining participants. The held-out
participant is then projected onto the training-derived dominant
transition axis.

This prevents the held-out participant from contributing to the
definition of the prediction target.

Because PCA component signs are arbitrary, the first component is
oriented consistently so that its mean loading is positive.

In [ ]:
from sklearn.decomposition import PCA

outer_loo = LeaveOneOut()

transition_true = []
transition_pred = []
transition_subjects = []
transition_alphas = []

for train_idx, test_idx in outer_loo.split(X_group):

    # -----------------------------
    # Predictor data
    # -----------------------------
    X_train = X_group[train_idx]
    X_test = X_group[test_idx]

    # -----------------------------
    # Reorganization profiles
    # -----------------------------
    Y_train = Y_delta[train_idx]
    Y_test = Y_delta[test_idx]

    # -----------------------------
    # PCA fitted ONLY on training participants
    # -----------------------------
    target_pca = PCA(n_components=1)

    train_scores = target_pca.fit_transform(
        Y_train
    ).ravel()

    test_score = target_pca.transform(
        Y_test
    ).ravel()[0]

    # Orient PC1 consistently
    if target_pca.components_[0].mean() < 0:
        train_scores *= -1
        test_score *= -1

    # -----------------------------
    # Ridge model
    # -----------------------------
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge())
    ])

    inner_loo = LeaveOneOut()

    search = GridSearchCV(
        pipeline,
        param_grid=alpha_grid,
        cv=inner_loo,
        scoring="neg_mean_absolute_error"
    )

    search.fit(
        X_train,
        train_scores
    )

    predicted_score = search.predict(
        X_test
    )[0]

    transition_true.append(
        test_score
    )

    transition_pred.append(
        predicted_score
    )

    transition_subjects.append(
        prediction_group_df.iloc[
            test_idx[0]
        ]["Subject"]
    )

    transition_alphas.append(
        search.best_params_["ridge__alpha"]
    )


transition_true = np.array(
    transition_true
)

transition_pred = np.array(
    transition_pred
)

transition_mae = mean_absolute_error(
    transition_true,
    transition_pred
)

transition_r, transition_p = pearsonr(
    transition_true,
    transition_pred
)

print(
    "LEAKAGE-FREE DOMINANT TRANSITION PREDICTION"
)

print(
    "\nMAE:",
    round(transition_mae, 4)
)

print(
    "Pearson r:",
    round(transition_r, 4)
)

print(
    "Correlation p:",
    round(transition_p, 4)
)

print("\nSelected alpha frequencies:")

print(
    pd.Series(
        transition_alphas
    )
    .value_counts()
    .sort_index()
)

In [ ]:
baseline_transition_true = []
baseline_transition_pred = []

for train_idx, test_idx in outer_loo.split(X_group):

    Y_train = Y_delta[train_idx]
    Y_test = Y_delta[test_idx]

    fold_pca = PCA(n_components=1)

    train_scores = fold_pca.fit_transform(
        Y_train
    ).ravel()

    test_score = fold_pca.transform(
        Y_test
    ).ravel()[0]

    # Same orientation rule
    if fold_pca.components_[0].mean() < 0:
        train_scores *= -1
        test_score *= -1

    baseline_prediction = np.mean(
        train_scores
    )

    baseline_transition_true.append(
        test_score
    )

    baseline_transition_pred.append(
        baseline_prediction
    )


baseline_transition_true = np.array(
    baseline_transition_true
)

baseline_transition_pred = np.array(
    baseline_transition_pred
)

baseline_transition_mae = mean_absolute_error(
    baseline_transition_true,
    baseline_transition_pred
)

print("DOMINANT TRANSITION BASELINE")

print(
    "\nBaseline MAE:",
    round(baseline_transition_mae, 4)
)

print(
    "Ridge MAE:",
    round(transition_mae, 4)
)

print(
    "\nMAE improvement:",
    round(
        baseline_transition_mae -
        transition_mae,
        4
    )
)

print(
    "Percent MAE improvement:",
    round(
        (
            baseline_transition_mae -
            transition_mae
        )
        /
        baseline_transition_mae
        * 100,
        2
    ),
    "%"
)

## 13 — Contribution of Intervention Information

To determine whether intervention assignment contributes predictive
information beyond baseline brain organization, the leakage-free
dominant-transition model is repeated using baseline Rest1
connectivity features alone.

Performance is compared with the intervention-aware model.

In [ ]:
connectivity_true = []
connectivity_pred = []
connectivity_alphas = []

for train_idx, test_idx in outer_loo.split(X):

    X_train = X[train_idx]
    X_test = X[test_idx]

    Y_train = Y_delta[train_idx]
    Y_test = Y_delta[test_idx]

    # PCA fitted only on training participants
    fold_pca = PCA(n_components=1)

    train_scores = fold_pca.fit_transform(
        Y_train
    ).ravel()

    test_score = fold_pca.transform(
        Y_test
    ).ravel()[0]

    # Consistent orientation
    if fold_pca.components_[0].mean() < 0:
        train_scores *= -1
        test_score *= -1

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge())
    ])

    search = GridSearchCV(
        pipeline,
        param_grid=alpha_grid,
        cv=LeaveOneOut(),
        scoring="neg_mean_absolute_error"
    )

    search.fit(
        X_train,
        train_scores
    )

    prediction = search.predict(
        X_test
    )[0]

    connectivity_true.append(
        test_score
    )

    connectivity_pred.append(
        prediction
    )

    connectivity_alphas.append(
        search.best_params_["ridge__alpha"]
    )


connectivity_true = np.array(
    connectivity_true
)

connectivity_pred = np.array(
    connectivity_pred
)

connectivity_mae = mean_absolute_error(
    connectivity_true,
    connectivity_pred
)

connectivity_r, connectivity_p = pearsonr(
    connectivity_true,
    connectivity_pred
)

print("CONNECTIVITY-ONLY MODEL")

print(
    "\nMAE:",
    round(connectivity_mae, 4)
)

print(
    "Pearson r:",
    round(connectivity_r, 4)
)

print(
    "Correlation p:",
    round(connectivity_p, 4)
)

print("\nINTERVENTION-AWARE MODEL")

print(
    "MAE:",
    round(transition_mae, 4)
)

print(
    "Pearson r:",
    round(transition_r, 4)
)

print(
    "\nMAE difference:",
    round(
        connectivity_mae -
        transition_mae,
        4
    )
)

## 14 — Permutation-Based Validation

Permutation testing was used to determine whether the observed
out-of-sample prediction performance exceeded that expected under
the null hypothesis of no relationship between baseline connectivity
and subsequent network reorganization.

For each permutation, participant reorganization profiles were
randomly reassigned relative to baseline connectivity. The complete
leakage-free prediction procedure, including training-fold PCA and
regularized regression, was then repeated.

The permutation p-value represents the proportion of null prediction
correlations equal to or greater than the observed out-of-sample
correlation.

In [ ]:
from sklearn.model_selection import LeaveOneOut, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
import numpy as np

# --------------------------------------------------
# Observed connectivity-only prediction correlation
# --------------------------------------------------

observed_r = connectivity_r

print(
    "Observed prediction r:",
    round(observed_r, 4)
)

# --------------------------------------------------
# Permutation settings
# --------------------------------------------------

N_PERMUTATIONS = 1000
rng = np.random.default_rng(42)

permutation_r = []

outer_loo = LeaveOneOut()

# --------------------------------------------------
# Permutation loop
# --------------------------------------------------

for perm in range(N_PERMUTATIONS):

    # Break participant-level correspondence between
    # baseline connectivity and subsequent reorganization
    permuted_indices = rng.permutation(
        len(Y_delta)
    )

    Y_perm = Y_delta[
        permuted_indices
    ]

    perm_true = []
    perm_pred = []

    for train_idx, test_idx in outer_loo.split(X):

        X_train = X[train_idx]
        X_test = X[test_idx]

        Y_train = Y_perm[train_idx]
        Y_test = Y_perm[test_idx]

        # ------------------------------------------
        # Define transition axis using training data only
        # ------------------------------------------

        fold_pca = PCA(
            n_components=1
        )

        train_scores = (
            fold_pca
            .fit_transform(Y_train)
            .ravel()
        )

        test_score = (
            fold_pca
            .transform(Y_test)
            .ravel()[0]
        )

        # Consistent PC1 orientation
        if fold_pca.components_[0].mean() < 0:
            train_scores *= -1
            test_score *= -1

        # ------------------------------------------
        # Nested Ridge model
        # ------------------------------------------

        pipeline = Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "ridge",
                Ridge()
            )
        ])

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=alpha_grid,
            cv=LeaveOneOut(),
            scoring="neg_mean_absolute_error"
        )

        search.fit(
            X_train,
            train_scores
        )

        prediction = search.predict(
            X_test
        )[0]

        perm_true.append(
            test_score
        )

        perm_pred.append(
            prediction
        )

    perm_true = np.array(
        perm_true
    )

    perm_pred = np.array(
        perm_pred
    )

    r_perm, _ = pearsonr(
        perm_true,
        perm_pred
    )

    permutation_r.append(
        r_perm
    )

    # Progress indicator
    if (perm + 1) % 100 == 0:
        print(
            f"Completed {perm + 1} / "
            f"{N_PERMUTATIONS}"
        )


permutation_r = np.array(
    permutation_r
)

# --------------------------------------------------
# One-sided permutation p-value
# --------------------------------------------------

permutation_p = (
    np.sum(
        permutation_r >= observed_r
    ) + 1
) / (
    N_PERMUTATIONS + 1
)

print(
    "\nPERMUTATION TEST"
)

print(
    "Observed r:",
    round(observed_r, 4)
)

print(
    "Null mean r:",
    round(
        np.mean(permutation_r),
        4
    )
)

print(
    "Null SD:",
    round(
        np.std(
            permutation_r,
            ddof=1
        ),
        4
    )
)

print(
    "Permutation p-value:",
    round(permutation_p, 4)
)

print(
    "Null correlations >= observed:",
    np.sum(
        permutation_r >= observed_r
    )
)

### Permutation-Test Interpretation

The connectivity-only model achieved an out-of-sample correlation of
r = 0.381 between predicted and observed dominant transition scores.

Permutation testing (1,000 permutations) produced a null mean
correlation of -0.119. Only 45 of 1,000 permutations produced a
correlation equal to or greater than the observed value, corresponding
to a permutation p-value of 0.046.

These results provide evidence that baseline resting-state connectivity
contains predictive information about the dominant direction of
subsequent network reorganization beyond that expected from random
participant correspondence. However, the predictive effect was modest,
and improvement in MAE relative to the baseline predictor was small.
Given the limited sample size, the result should be interpreted as
preliminary rather than as evidence of a clinically validated
predictive biomarker.

In [ ]:
# Inspect prediction-related variables already in memory
for name in sorted(globals()):
    if any(word in name.lower() for word in
           ["ridge", "coef", "alpha", "pred", "fold", "feature", "outer"]):
        obj = globals()[name]
        try:
            shape = obj.shape
        except:
            shape = None
        print(f"{name:35s} {type(obj).__name__:20s} {shape}")

In [ ]:
print("feature_columns:")
print(feature_columns)

print("\nalpha_grid:")
print(alpha_grid)

print("\nconnectivity_alphas:")
print(connectivity_alphas)

print("\nShapes:")
print("X:", X.shape if "X" in globals() else "X not found")
print("X_connectivity:",
      X_connectivity.shape if "X_connectivity" in globals()
      else "X_connectivity not found")
print("Y_delta:", Y_delta.shape)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()

fold_coefficients = []
reconstructed_pred = np.zeros(len(X))

for fold, (train_idx, test_idx) in enumerate(loo.split(X)):

    # -------------------------------------------------
    # 1. Fit target PCA using TRAINING participants only
    # -------------------------------------------------
    pca_fold = PCA(n_components=1)

    y_train = pca_fold.fit_transform(
        Y_delta[train_idx]
    ).ravel()

    y_test = pca_fold.transform(
        Y_delta[test_idx]
    ).ravel()

    # Orient PC1 consistently so average loading is positive
    if pca_fold.components_[0].mean() < 0:
        y_train *= -1
        y_test *= -1

    # -------------------------------------------------
    # 2. Standardize baseline connectivity using
    #    TRAINING participants only
    # -------------------------------------------------
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X[train_idx])
    X_test_scaled = scaler.transform(X[test_idx])

    # -------------------------------------------------
    # 3. Use alpha already selected by your nested CV
    # -------------------------------------------------
    alpha = connectivity_alphas[fold]

    model = Ridge(alpha=alpha)

    model.fit(
        X_train_scaled,
        y_train
    )

    reconstructed_pred[test_idx[0]] = model.predict(
        X_test_scaled
    )[0]

    # Save standardized coefficients
    fold_coefficients.append(model.coef_.copy())


fold_coefficients = np.vstack(fold_coefficients)

print("Coefficient matrix shape:",
      fold_coefficients.shape)

print("\nOriginal predictions:")
print(np.round(connectivity_pred, 4))

print("\nReconstructed predictions:")
print(np.round(reconstructed_pred, 4))

print(
    "\nMax absolute prediction difference:",
    np.max(
        np.abs(
            connectivity_pred -
            reconstructed_pred
        )
    )
)

print(
    "Predictions match:",
    np.allclose(
        connectivity_pred,
        reconstructed_pred,
        atol=1e-8
    )
)

In [ ]:
coef_df = pd.DataFrame(
    fold_coefficients,
    columns=feature_columns
)

stability_results = pd.DataFrame({
    "Feature": feature_columns,
    "Mean_Coefficient": coef_df.mean(axis=0).values,
    "Mean_Abs_Coefficient": coef_df.abs().mean(axis=0).values,
    "SD_Coefficient": coef_df.std(axis=0).values,
})

# Sign consistency:
# proportion of folds having the same sign as the mean coefficient
sign_consistency = []

for feature in feature_columns:
    values = coef_df[feature].values
    mean_sign = np.sign(values.mean())

    if mean_sign == 0:
        consistency = 0
    else:
        consistency = np.mean(np.sign(values) == mean_sign)

    sign_consistency.append(consistency)

stability_results["Sign_Consistency"] = sign_consistency

# Rank primarily by mean absolute standardized coefficient
stability_results = stability_results.sort_values(
    "Mean_Abs_Coefficient",
    ascending=False
).reset_index(drop=True)

print("Top 15 baseline connectivity contributors:")
print(
    stability_results.head(15).round(4).to_string(index=False)
)

In [ ]:
# Inspect whether the two low-regularization folds dominate coefficient magnitudes

fold_summary = pd.DataFrame({
    "Fold": np.arange(1, 19),
    "Alpha": connectivity_alphas,
    "Coefficient_L2_Norm": np.linalg.norm(fold_coefficients, axis=1),
    "Mean_Abs_Coefficient": np.mean(np.abs(fold_coefficients), axis=1)
})

print(fold_summary.round(4).to_string(index=False))

print("\nMean coefficient magnitude by selected alpha:")
print(
    fold_summary.groupby("Alpha")[
        ["Coefficient_L2_Norm", "Mean_Abs_Coefficient"]
    ].mean().round(4)
)

In [ ]:
# Normalize absolute coefficient importance within each outer fold
abs_coef = np.abs(fold_coefficients)

relative_importance = (
    abs_coef /
    abs_coef.sum(axis=1, keepdims=True)
)

relative_importance_df = pd.DataFrame(
    relative_importance,
    columns=feature_columns
)

# Rank features by average relative importance across folds
relative_results = pd.DataFrame({
    "Feature": feature_columns,
    "Mean_Relative_Importance":
        relative_importance_df.mean(axis=0).values,

    "SD_Relative_Importance":
        relative_importance_df.std(axis=0).values,

    "Median_Relative_Importance":
        relative_importance_df.median(axis=0).values
})

relative_results = relative_results.sort_values(
    "Mean_Relative_Importance",
    ascending=False
).reset_index(drop=True)

print("Top 15 features by fold-normalized importance:")
print(
    relative_results.head(15)
    .round(4)
    .to_string(index=False)
)

print(
    "\nTotal mean relative importance:",
    relative_results["Mean_Relative_Importance"].sum()
)

In [ ]:
print("PCA-related variables:\n")

for name in sorted(globals()):
    if any(word in name.lower() for word in
           ["pca", "explained", "loading", "score", "transition"]):
        obj = globals()[name]
        try:
            shape = obj.shape
        except:
            shape = None

        print(f"{name:35s} {type(obj).__name__:20s} {shape}")

In [ ]:
print("Explained variance:")
print(np.round(explained[:10], 4))

print("\npc1_loadings columns:")
print(pc1_loadings.columns.tolist())

print("\nTop rows of pc1_loadings:")
print(pc1_loadings.head(10).to_string(index=False))

print("\npc1_scores_df:")
print(pc1_scores_df.head().to_string(index=False))

In [ ]:
# =======================================================
# FIGURE 5
# Dominant brain-network transition pattern
# =======================================================

import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------
# Panel A: PCA variance explained
# -------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5),
    gridspec_kw={"width_ratios": [0.8, 1.2]}
)

ax = axes[0]

pcs_to_show = 10
x = np.arange(1, pcs_to_show + 1)

ax.bar(
    x,
    explained[:pcs_to_show] * 100
)

ax.plot(
    x,
    np.cumsum(explained[:pcs_to_show]) * 100,
    marker="o"
)

ax.set_xlabel("Principal component")
ax.set_ylabel("Variance explained (%)")
ax.set_xticks(x)

ax.set_title(
    "(A) PCA of Network-Transition Features"
)

ax.axhline(
    80,
    linestyle="--",
    linewidth=1
)

# Annotate PC1
ax.text(
    1,
    explained[0] * 100 + 3,
    f"{explained[0]*100:.1f}%",
    ha="center"
)

# -------------------------------------------------------
# Panel B: PC1 loadings
# -------------------------------------------------------

ax = axes[1]

loading_plot = (
    pc1_loadings
    .sort_values(
        "Absolute_Loading",
        ascending=True
    )
    .tail(15)
    .copy()
)

# Cleaner labels
loading_plot["Clean_Feature"] = (
    loading_plot["Feature"]
    .str.replace("Delta_", "", regex=False)
    .str.replace("__", "–", regex=False)
)

ax.barh(
    loading_plot["Clean_Feature"],
    loading_plot["PC1_Loading"]
)

ax.axvline(
    0,
    linewidth=1
)

ax.set_xlabel("PC1 loading")
ax.set_title(
    "(B) Largest PC1 Network-Pair Loadings"
)

plt.tight_layout()

plt.savefig(
    "Fig5_Dominant_Transition_PCA.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Fig5_Dominant_Transition_PCA.svg",
    bbox_inches="tight"
)

plt.show()

print("PC1 variance explained:",
      round(explained[0] * 100, 1), "%")

print("Cumulative variance, first 3 PCs:",
      round(np.sum(explained[:3]) * 100, 1), "%")

In [ ]:
print("Prediction arrays:")
print("transition_true:",
      transition_true.shape,
      np.round(transition_true, 4))

print("\nconnectivity_pred:",
      connectivity_pred.shape,
      np.round(connectivity_pred, 4))

print("\nPermutation-related variables:")
for name in sorted(globals()):
    if any(word in name.lower() for word in
           ["perm", "null", "importance", "relative"]):
        obj = globals()[name]
        try:
            shape = obj.shape
        except:
            shape = None

        print(f"{name:35s} {type(obj).__name__:20s} {shape}")

print("\nRelative importance table available:",
      "relative_results" in globals())

if "relative_results" in globals():
    print(relative_results.head(10).to_string(index=False))

In [ ]:
# =======================================================
# FIGURE 6
# Prediction, permutation validation, and model interpretation
# =======================================================

import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.8),
    gridspec_kw={"width_ratios": [1, 1, 1.25]}
)

# -------------------------------------------------------
# Panel A: Observed vs predicted dominant transition
# -------------------------------------------------------

ax = axes[0]

ax.scatter(
    transition_true,
    connectivity_pred,
    s=45
)

# identity line for reference
lims = [
    min(transition_true.min(), connectivity_pred.min()),
    max(transition_true.max(), connectivity_pred.max())
]

ax.plot(
    lims,
    lims,
    linestyle="--",
    linewidth=1
)

ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("Observed dominant transition score")
ax.set_ylabel("Predicted dominant transition score")

ax.set_title("(A) Out-of-Sample Prediction")

ax.text(
    0.05,
    0.95,
    r"$r=0.381$" + "\n" + r"MAE $=0.369$",
    transform=ax.transAxes,
    va="top"
)

# -------------------------------------------------------
# Panel B: permutation null distribution
# -------------------------------------------------------

ax = axes[1]

ax.hist(
    permutation_r,
    bins=30
)

ax.axvline(
    0.3814,
    linestyle="--",
    linewidth=2
)

ax.set_xlabel("Prediction correlation under permutation")
ax.set_ylabel("Frequency")

ax.set_title("(B) Permutation Validation")

ax.text(
    0.05,
    0.95,
    r"Observed $r=0.381$" + "\n" + r"$p_{\mathrm{perm}}=0.046$",
    transform=ax.transAxes,
    va="top"
)

# -------------------------------------------------------
# Panel C: fold-normalized feature importance
# -------------------------------------------------------

ax = axes[2]

importance_plot = (
    relative_results
    .sort_values(
        "Mean_Relative_Importance",
        ascending=True
    )
    .tail(10)
    .copy()
)

importance_plot["Clean_Feature"] = (
    importance_plot["Feature"]
    .str.replace("__", "–", regex=False)
)

ax.barh(
    importance_plot["Clean_Feature"],
    importance_plot["Mean_Relative_Importance"]
)

ax.set_xlabel("Mean fold-normalized relative importance")
ax.set_title("(C) Baseline Model Contributions")
plt.tight_layout()

plt.savefig(
    "Fig6_Prediction_and_Interpretation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Fig6_Prediction_and_Interpretation.svg",
    bbox_inches="tight"
)

plt.show()